In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
dbutils.widgets.text("environment", "dev", "Deployment Environment")
env = dbutils.widgets.get("environment")
catalog = f'{env}_banking'
reporting_schema = 'platinum'
dwh_schema='gold'

dbutils.widgets.text("as_of_date", "2024-12-31", "As-of business date")
dbutils.widgets.text("from_date", "2024-11-01", "Report start")
dbutils.widgets.text("to_date", "2024-12-31", "Report end")

as_of_date, from_date, to_date = [
    F.to_date(F.lit(dbutils.widgets.get(x))) for x in ["as_of_date", "from_date", "to_date"]
]
print(env, as_of_date, from_date, to_date)

In [0]:

def full_table_name(schema, name):
    return f"{catalog}.{schema}.{name}"

date = spark.table(full_table_name(dwh_schema, 'dim_date'))
branch = spark.table(full_table_name(dwh_schema, 'dim_branch'))
customer = spark.table(full_table_name(dwh_schema, 'dim_customer'))
transactions = spark.table(full_table_name(dwh_schema, 'fact_transactions'))
balance = spark.table(full_table_name(dwh_schema, 'fact_account_balance'))
ownership = spark.table(full_table_name(dwh_schema, 'bridge_account_customer'))
product = spark.table(full_table_name(dwh_schema, 'dim_product'))
transaction_product = spark.table(full_table_name(dwh_schema, 'bridge_transaction_product'))

# Historical banking reporting from DWH

This notebook creates governed reporting datasets and runs point-in-time metrics with PySpark DataFrames. Set `as_of_date` to a historical business date such as `2024-02-20`; every customer and ownership attribute is resolved as of that date, while facts are filtered by business dates rather than ingestion dates.

## 1. Reporting views

Views keep metric definitions reusable and prevent BI users from joining raw fact grains incorrectly. Ownership allocations are explicit, so joint-account balances and amounts can be analyzed per owner without changing fact grain.

In [0]:
transaction_yearly = (
    transactions.join(date, "DateSK")
    .join(branch, "BranchSK").drop('State')
    .join(customer, "CustomerSK")
    .groupBy(
        "YearNumber", "BranchName", "CustomerSegment", "State"
    ).agg(
        F.count("*").alias("TransactionCount"),
        F.sum(F.when(F.col("TransactionStatus") == "POSTED", F.col("TransactionAmount")).otherwise(0)).alias("PostedNetAmount"),
        F.sum(F.when((F.col("TransactionType") == "DEPOSIT")& (F.col("TransactionStatus") == "POSTED"),F.col("TransactionAmount"),).otherwise(0)).alias("DepositAmount"),
        F.sum(F.when(F.col("TransactionType").isin("WITHDRAWAL", "DEBIT") & (F.col("TransactionStatus") == "POSTED"),F.abs(F.col("TransactionAmount")),).otherwise(0)).alias("DebitAmount"),
        F.sum(F.when(F.col("TransactionStatus") == "REVERSED", 1).otherwise(0)).alias("ReversalCount")
    )
)

transaction_yearly.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    full_table_name(reporting_schema,"transaction_yearly")
)

In [0]:
transaction_yearly.sort('YearNumber', 'BranchName', 'CustomerSegment', 'State').show()

In [0]:
balance_as_of = balance.join(customer, "CustomerSK").select( 
                    "BalanceDate", "AccountID", "BranchSK", "CustomerSK", "CurrencyCode", "AvailableBalanceAmount", "LedgerBalanceAmount", "OverdraftLimitAmount", "CustomerID", "CustomerSegment", "KYCStatus", "State")

# balance_as_of.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(full_table_name(reporting_schema, "rpt_balance_as_of"))


In [0]:
owner_allocated_activity = (
    transactions.join(date, "DateSK")
    .join(ownership, "AccountID")
    .join(customer, ownership.CustomerSK == customer.CustomerSK)
    .filter((F.to_date(F.col("CalendarDate")) >= F.col("StartDate")) & (F.to_date(F.col("CalendarDate")) < F.col("EndDate")))
    .select( "DateSK", "CalendarDate", "AccountID", "ownership.CustomerID", "customer.FullName", "customer.CustomerSegment", "ownership.OwnershipPercent",
        (F.col("TransactionAmount") * F.col("OwnershipPercent") / 100).alias("AllocatedTransactionAmount")))

# owner_allocated_activity.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(reporting_schema("rpt_owner_allocated_activity"))

## 2. As-of portfolio and customer metrics

The first DataFrame is a point-in-time snapshot for `as_of_date`. The second gives the historical daily trend through that date. Replace widget values in a job or dashboard without changing the transformation logic.

In [0]:
latest_balance = (
    balance.filter(F.col("BalanceDate") <= as_of_date)
    .withColumn("_rn", F.row_number().over(Window.partitionBy("AccountID").orderBy(F.col("BalanceDate").desc()))).filter(F.col("_rn") == 1).drop("_rn")
)
asof_owner = ownership.filter((as_of_date >= F.col("StartDate")) & (as_of_date < F.col("EndDate")))
asof_customer = customer.filter((as_of_date >= F.col("StartDate")) & (as_of_date < F.col("EndDate")))

portfolio_position = (
    latest_balance.join(asof_owner, "AccountID")
    .join(asof_customer, asof_owner.CustomerID == asof_customer.CustomerID)
    .withColumn('LedgerBalanceAmount', F.expr('LedgerBalanceAmount * OwnershipPercent / 100')) # Balance the LedgerBalanceAmount by the account's ownership percentage
    .withColumn('AvailableBalanceAmount', F.expr('AvailableBalanceAmount * OwnershipPercent / 100')) # Balance the AvailableBalanceAmount by the account's ownership percentage
    .groupBy(asof_customer.CustomerSegment, asof_customer.State)
    .agg(
        F.countDistinct("AccountID").alias("AccountCount"),
        F.sum("LedgerBalanceAmount").alias("LedgerBalance"),
        F.sum("AvailableBalanceAmount").alias("AvailableBalance"),
        F.sum(F.greatest(-F.col("LedgerBalanceAmount"), F.lit(0))).alias("OverdraftExposure"),
    )
    .orderBy("CustomerSegment", "State")
)
display(portfolio_position)


In [0]:
period_transactions = (
    transactions.join(date, "DateSK").join(branch, "BranchSK").join(customer, "CustomerSK")
        .filter((F.col("CalendarDate") >= from_date) & (F.col("CalendarDate") <= to_date))
)

daily_trend = (
    period_transactions.groupBy("CalendarDate", "BranchName")
    .agg(F.count("*").alias("TransactionCount"),
        F.sum(F.when(F.col("TransactionStatus") == "POSTED", F.col("TransactionAmount")).otherwise(0)).alias("NetPostedAmount"),
        F.sum(F.when(F.col("TransactionType") == "DIRECT_DEPOSIT", F.col("TransactionAmount")).otherwise(0)).alias("DepositAmount"),
        F.sum(F.when(F.col("TransactionType") == "ACH_DEBIT", -F.col("TransactionAmount")).otherwise(0)).alias("DebitAmount"),
        F.sum(F.when(F.col("TransactionType") == "ATM_WITHDRAWAL", -F.col("TransactionAmount")).otherwise(0)).alias("ATMWithdrawlAmount"),
        F.sum(F.when(F.col("TransactionType") == "TRANSFER", -F.col("TransactionAmount")).otherwise(0)).alias("TransferAmount"), 
        F.sum(F.when(F.col("TransactionStatus") == "REVERSED", 1).otherwise(0)).alias("ReversalCount"),
    )
    .orderBy("CalendarDate", "BranchName")
)
display(daily_trend)

## 3. As-of ownership allocation and product reporting

Use the allocation query for joint accounts. It allocates each atomic transaction and account-day balance using the ownership version valid on the selected date. Product analysis stays at line grain and therefore never multiplies transaction totals.

In [0]:
allocated_activity = (
    transactions.join(date, "DateSK")
    .join(asof_owner, "AccountID")
    .join(asof_customer, asof_owner.CustomerID == asof_customer.CustomerID)
    .filter((F.col("CalendarDate") <= as_of_date) & (F.col("TransactionStatus") == "POSTED"))
    .groupBy(asof_owner.CustomerID, asof_customer.FullName, asof_customer.CustomerSegment)
    .agg(
        F.sum(F.col("TransactionAmount")).alias("NetActivity"),
        F.sum(F.col("TransactionAmount") * F.col("OwnershipPercent") / 100).alias("AllocatedNetActivity"),
        F.countDistinct("TxnID").alias("TransactionCount"),
    )
)
display(allocated_activity)

In [0]:
product_activity = (
    transaction_product.join(product, "ProductSK")
    .join(transactions, "TxnID")
    .join(date, "DateSK")
    .filter(
        (F.col("CalendarDate") >= from_date) & (F.col("CalendarDate") <= as_of_date)
    )
    .groupBy("ProductCategory", "ProductName")
    .agg(
        F.sum("Quantity").alias("TotalQuantity"),
        F.sum("LineAmount").alias("ProductRevenue"),
        F.countDistinct("TxnID").alias("Transactions"),
    )
)
display(product_activity)